In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
import os, sys

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import scrublet as scr

sc.settings.verbosity = 3
sc.logging.print_header()
sc.settings.set_figure_params(dpi=100, figsize=(5,5))

In [2]:
# Path designation
download_path = "/home/neuro_demo_research/data/downloads/"
save_path = "/home/neuro_demo_research/data/active/biml_tutorial"


In [8]:
# 3. Downloading
pbmc_5k = sc.read_10x_mtx(
    download_path + "pbmc5k/outs/filtered_feature_bc_matrix",
    cache=True
)

pbmc_10k = sc.read_10x_mtx(
    download_path + "pbmc10k/outs/filtered_feature_bc_matrix",
    cache=True
)

... reading from cache file cache/home-neuro_demo_research-data-downloads-pbmc5k-outs-filtered_feature_bc_matrix-matrix.h5ad


... reading from cache file cache/home-neuro_demo_research-data-downloads-pbmc10k-outs-filtered_feature_bc_matrix-matrix.h5ad


In [9]:
# 4. Doublet detection
pbmc_5k.obs['doublet_scores'], pbmc_5k.obs['predicted_doublets'] = scr.Scrublet(pbmc_5k.X).scrub_doublets(verbose=False)
pbmc_5k.obs['predicted_doublets'] = pbmc_5k.obs['predicted_doublets'].astype(str)

In [10]:
pbmc_10k.obs['doublet_scores'], pbmc_10k.obs['predicted_doublets'] = scr.Scrublet(pbmc_10k.X).scrub_doublets(verbose=False)
pbmc_10k.obs['predicted_doublets'] = pbmc_10k.obs['predicted_doublets'].astype(str)

In [12]:
pbmc_5k.obs

,doublet_scores,predicted_doublets
AAACCCAAGCGTATGG-1,0.049904,False
AAACCCAGTCCTACAA-1,0.040210,False
AAACCCATCACCTCAC-1,0.025223,False
AAACGCTAGGGCATGT-1,0.034653,False
AAACGCTGTAGGTACG-1,0.051707,False
...,...,...
TTTGTTGCAGGTACGA-1,0.286585,False
TTTGTTGCAGTCTCTC-1,0.119874,False
TTTGTTGGTAATTAGG-1,0.015810,False
TTTGTTGTCCTTGGAA-1,0.037351,False


In [17]:
# 5. Merging two data
pbmc_5k

AnnData object with n_obs × n_vars = 5025 × 33538
    obs: 'doublet_scores', 'predicted_doublets'
    var: 'gene_ids', 'feature_types'

In [18]:
pbmc_10k

AnnData object with n_obs × n_vars = 11769 × 33538
    obs: 'doublet_scores', 'predicted_doublets'
    var: 'gene_ids', 'feature_types'

In [19]:
pbmc_5k.var.index.is_unique

True

In [20]:
pbmc_10k.var.index.is_unique

True

In [21]:
pbmc_5k.obs['dataset'] = '5K PBMC'
pbmc_10k.obs['dataset'] = '10K PBMC'

In [22]:
pbmc_5k.obs.index.is_unique

True

In [23]:
pbmc_10k.obs.index.is_unique

True

In [24]:
adata = sc.concat([pbmc_5k, pbmc_10k], merge='same')

/home/neuro_demo_research/miniconda3/envs/biml/lib/python3.10/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [26]:
adata.obs.index.is_unique

False

In [27]:
adata.var.index.is_unique

True

In [28]:
adata.obs_names_make_unique()

In [29]:
adata.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 36195319 stored elements and shape (16794, 33538)>

In [30]:
adata.obs

,doublet_scores,predicted_doublets,dataset
AAACCCAAGCGTATGG-1,0.049904,False,5K PBMC
AAACCCAGTCCTACAA-1,0.040210,False,5K PBMC
AAACCCATCACCTCAC-1,0.025223,False,5K PBMC
AAACGCTAGGGCATGT-1,0.034653,False,5K PBMC
AAACGCTGTAGGTACG-1,0.051707,False,5K PBMC
...,...,...,...
TTTGTTGGTGTCATGT-1,0.124736,False,10K PBMC
TTTGTTGGTTTGAACC-1,0.022472,False,10K PBMC
TTTGTTGTCCAAGCCG-1,0.118367,False,10K PBMC
TTTGTTGTCTTACTGT-1,0.043684,False,10K PBMC


In [31]:
adata.var

,gene_ids,feature_types
MIR1302-2HG,ENSG00000243485,Gene Expression
FAM138A,ENSG00000237613,Gene Expression
OR4F5,ENSG00000186092,Gene Expression
AL627309.1,ENSG00000238009,Gene Expression
AL627309.3,ENSG00000239945,Gene Expression
...,...,...
AC233755.2,ENSG00000277856,Gene Expression
AC233755.1,ENSG00000275063,Gene Expression
AC240274.1,ENSG00000271254,Gene Expression
AC213203.1,ENSG00000277475,Gene Expression


In [36]:
adata.write(
    save_path + "/adata_raw"
)

In [ ]:
#adata = sc.read_h5ad(save_path + "/adata_raw")

In [5]:
# 6. Preprocessing

adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, inplace=True)

In [6]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True, 
             save="_violin.png")

In [ ]:
#plt.close('all')
## 이거 안하고 하면 plot 여러 개 겹쳐서 나옴

In [7]:
plt.scatter(
    adata.obs['log1p_total_counts'], 
    adata.obs['log1p_n_genes_by_counts'], 
    c=adata.obs['pct_counts_mt'], 
    s=0.01
)
plt.axvline(x=np.log1p(2000), color='r', linestyle='--', linewidth=1) # min_counts
plt.axhline(y=np.log1p(500), color='r', linestyle='--', linewidth=1) # min_genes
plt.axhline(y=np.log1p(7000), color='r', linestyle='--', linewidth=1) # max_genes
plt.xlabel('log1p_total_counts')
plt.ylabel('log1p_n_genes_by_counts')
plt.colorbar()
plt.savefig("scatter.png", dpi=300, bbox_inches='tight')
plt.show()

In [8]:
# Filtering
sc.pp.filter_cells(adata, min_counts=2000)
sc.pp.filter_cells(adata, min_genes=500)
sc.pp.filter_cells(adata, max_genes=7000)
adata = adata[adata.obs['pct_counts_mt'] < 20, :]
adata = adata[adata.obs['predicted_doublets'] == 'False', :]

filtered out 1165 cells that have less than 2000 counts


filtered out 10 cells that have less than 500 genes expressed
filtered out 1 cells that have more than 7000 genes expressed


In [9]:
print('before log1p norm')
print('max: ', np.max(adata.X))
print('min: ', np.min(adata.X))
print('mean: ', np.mean(adata.X))

before log1p norm
max:  17480.0
min:  0.0
mean:  0.24698788


In [10]:
adata.layers['counts'] = adata.X.copy()

In [11]:
sc.pp.normalize_total(adata, target_sum=1e4)

normalizing counts per cell
    finished (0:00:03)


In [12]:
adata.X.sum(axis=1)

matrix([[ 9999.999],
        [10000.   ],
        [ 9999.999],
        ...,
        [10000.   ],
        [10000.001],
        [10000.   ]], dtype=float32)

In [13]:
# Logarithmize
sc.pp.log1p(adata)
adata.raw = adata

In [14]:
print('after log1p norm')
print('max: ', np.max(adata.X))
print('min: ', np.min(adata.X))
print('mean: ', np.mean(adata.X))

after log1p norm
max:  8.463504
min:  0.0
mean:  0.0771922


In [15]:
# HVG
sc.pp.highly_variable_genes(adata)
sc.pl.highly_variable_genes(adata, save="_hvg.png")
print(f"detected {np.sum(adata.var['highly_variable'])} highly variable genes")

extracting highly variable genes
    finished (0:00:00)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)
detected 1885 highly variable genes


In [16]:
adata.var

,gene_ids,feature_types,mt,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts,highly_variable,means,dispersions,dispersions_norm
MIR1302-2HG,ENSG00000243485,Gene Expression,False,0,0.000000,0.000000,100.000000,0.0,0.000000,False,1.000000e-12,NaN,NaN
FAM138A,ENSG00000237613,Gene Expression,False,0,0.000000,0.000000,100.000000,0.0,0.000000,False,1.000000e-12,NaN,NaN
OR4F5,ENSG00000186092,Gene Expression,False,0,0.000000,0.000000,100.000000,0.0,0.000000,False,1.000000e-12,NaN,NaN
AL627309.1,ENSG00000238009,Gene Expression,False,109,0.006729,0.006706,99.350959,113.0,4.736198,False,9.053122e-03,0.576632,0.366783
AL627309.3,ENSG00000239945,Gene Expression,False,7,0.000417,0.000417,99.958318,7.0,2.079442,False,4.664992e-04,0.207318,-0.702617
...,...,...,...,...,...,...,...,...,...,...,...,...,...
AC233755.2,ENSG00000277856,Gene Expression,False,3,0.000179,0.000179,99.982136,3.0,1.386294,False,2.799133e-04,0.761968,0.903447
AC233755.1,ENSG00000275063,Gene Expression,False,4,0.000298,0.000298,99.976182,5.0,1.791759,False,8.087420e-04,1.677779,3.555303
AC240274.1,ENSG00000271254,Gene Expression,False,184,0.011611,0.011544,98.904371,195.0,5.278115,False,1.270118e-02,0.271960,-0.515437
AC213203.1,ENSG00000277475,Gene Expression,False,0,0.000000,0.000000,100.000000,0.0,0.000000,False,1.000000e-12,NaN,NaN


In [17]:
# scaling
sc.pp.scale(adata, max_value=10)

/home/neuro_demo_research/miniconda3/envs/biml/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


In [18]:
print('max: ', np.max(adata.X))
print('min: ', np.min(adata.X))
print('mean: ', np.mean(adata.X))

max:  10.0
min:  -10.0
mean:  -0.005574487703838729


In [ ]:
## cell cycle gene은 variance가 큼 (S vs G2M vs rest) -> PCA에서 cell cycle axis가 강하게 잡힘
## cell cycle gene을 hvg에서 제거하는데 나는 그게 없어서 pass

In [19]:
adata.write(
    save_path + "/adata_qc"
)

In [20]:
# 7. PCA, Harmony, UMAP

sc.tl.pca(adata)
sc.pl.pca_variance_ratio(adata, save="_pca.png")

computing PCA
    with n_comps=50


/tmp/ipykernel_2471/3335368604.py:3: UserWarning: When using a mask parameter with anndata<0.9 on a dense array, the PCAcan have slightly different results due the array being column major instead of row major.
  sc.tl.pca(adata)


    finished (0:00:03)


In [ ]:
#UMAP
sc.pp.neighbors(adata, n_pcs=10)
sc.tl.umap(adata)
sc.pl.umap(adata, color = 'dataset', save='dataset_npcs=10.png')

computing neighbors
    using 'X_pca' with n_pcs = 10
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:49)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:10)


In [24]:
sc.pp.neighbors(adata, n_pcs=20)
sc.tl.umap(adata)
sc.pl.umap(adata, color = 'dataset', save='dataset_npcs=20.png')

computing neighbors
    using 'X_pca' with n_pcs = 20
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:02)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:09)


In [25]:
sc.pp.neighbors(adata, n_pcs=50)
sc.tl.umap(adata)
sc.pl.umap(adata, color = 'dataset', save='dataset_npcs=50.png')

computing neighbors
    using 'X_pca' with n_pcs = 50
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:03)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:10)
